In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import proplot as pplt
from matplotlib.lines import Line2D
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})


In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
WEIGHTSDIR = CONFIGS['filepaths']['weights']
PREDSDIR   = CONFIGS['filepaths']['predictions']
SEEDS      = CONFIGS['experiments']['nn']['seeds']
FIELDVARS  = CONFIGS['experiments']['sr']['runs']['sr_gauss']['fieldvars']
LOCALVARS  = CONFIGS['experiments']['sr']['runs']['sr_gauss_resid']['localvars']
SPLIT      = 'test'

with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
TP_MEAN = STATS['tp_mean']
TP_STD  = STATS['tp_std']
ZMIN    = (0.0-TP_MEAN)/TP_STD

ALLMODELS = {**CONFIGS['experiments']['nn']['runs'],**CONFIGS['experiments']['sr']['optimizedeqs']}
ORDER     = ['sr_med','sr_hi','nn_gauss','sr_resid_lin','sr_resid_cube','sr_resid_full']
COLORS    = {name:ALLMODELS[name]['color'] for name in ORDER}
LABELS    = {name:ALLMODELS[name]['description'] for name in ORDER}
FORMS     = {name:ALLMODELS[name].get('form') for name in ORDER}
INITS     = {name:ALLMODELS[name].get('init',{}) for name in ORDER}


In [ ]:
SRFN = dict(cube=lambda x:x**3,square=lambda x:x**2,neg=lambda x:-x,
            exp=np.exp,log=np.log,abs=np.abs,sqrt=np.sqrt,
            max=np.maximum,min=np.minimum)

def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w *= mask[:,None,:]
    return w.sum(axis=2)

def eval_form(form,feats,constants=None):
    ns = dict(SRFN,__builtins__={},**feats)
    if constants is not None: ns.update(constants)
    return np.asarray(eval(form,ns),dtype=float)

def sr_to_mm(zresid):
    z = ZMIN+np.maximum(zresid,0.0)
    return np.maximum(np.expm1(z*TP_STD+TP_MEAN),0.0)

def to_da(arr,ref):
    return xr.DataArray(arr.reshape(ref.shape),dims=ref.dims,coords=ref.coords)

def get_r2(obs,pred):
    obs,pred = xr.align(obs,pred,join='inner')
    ssres = ((obs-pred)**2).sum(skipna=True)
    sstot = ((obs-obs.mean(skipna=True))**2).sum(skipna=True)
    return float(1-ssres/sstot)


In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime,nlat,nlon = ds.time.size,ds.lat.size,ds.lon.size
    nsig = ds.sizes.get('sig',1)
    dsig = ds.dsig.values
    refda = ds.tp.transpose('time','lat','lon').load()
    fields = np.stack([ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS],axis=1)
    surfmask = ds.surfmask.transpose('time','lat','lon','sig').values.reshape(-1,nsig) if 'surfmask' in ds else None
    flat = lambda v: ds[v].transpose('time','lat','lon').values.ravel() if 'time' in ds[v].dims else np.tile(ds[v].values,(ntime,1,1)).ravel()
    locals_ = {v:flat(v) for v in LOCALVARS}

kernels = [xr.open_dataset(os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc'),engine='h5netcdf')['k'].values for seed in SEEDS]
integrals = kernel_integrate(fields,np.mean(kernels,axis=0),dsig,surfmask)
FEATS = {**{v:integrals[:,i] for i,v in enumerate(FIELDVARS)},**locals_}

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    OBS = ds.tp.load()

with xr.open_dataset(os.path.join(PREDSDIR,f'nn_gauss_{SPLIT}_predictions.nc'),engine='h5netcdf') as ds:
    NN_GAUSS = ds.tp.mean('seed').load()

FEATS['srmed'] = eval_form(FORMS['sr_med'],FEATS,INITS['sr_med'])


In [ ]:
preds = {}
for name in ORDER:
    if name=='nn_gauss':
        preds[name] = NN_GAUSS
    else:
        preds[name] = to_da(sr_to_mm(eval_form(FORMS[name],FEATS,INITS[name])),refda)

r2 = {name:get_r2(OBS,preds[name]) for name in ORDER}
for name in ORDER:
    print(f'{LABELS[name]:14s}  R² = {r2[name]:.4f}')


In [ ]:
order = sorted(ORDER,key=lambda n:r2[n])
labels = [LABELS[n] for n in order]
values = [r2[n]    for n in order]
colors = [COLORS[n] for n in order]

fig,ax = pplt.subplots(figwidth=4,figheight=2.5)
ax.barh(labels,values,color=colors,alpha=.85,edgecolor='none')
ax.axvline(r2['sr_med'],  color=COLORS['sr_med'],  linestyle='--',linewidth=.7,zorder=0)
ax.axvline(r2['nn_gauss'],color=COLORS['nn_gauss'],linestyle='--',linewidth=.7,zorder=0)
for i,v in enumerate(values):
    ax.text(v+.005,i,f'{v:.3f}',va='center',ha='left',fontsize=8)
ax.format(grid=False,xlabel=r'$R^2$',xlim=(0,r2['nn_gauss']+.12),
          title='Consensus Residual Forms vs. Baselines')
pplt.show()


In [ ]:
summary = pd.DataFrame([
    dict(Model='SR-MED',       Complexity=16,Form='a·cube(max(rh, θe − b·θe★ − c))'),
    dict(Model='SR-HI',        Complexity=23,Form='cube(a·max(rh, θe + b·θe★ + c) + max(lf,shf)·d)'),
    dict(Model='NN-GAUSS',     Complexity='—',Form='Gaussian-kernel neural network'),
    dict(Model='SR-RESID-LIN', Complexity=20,Form='srmed + a·min(lf,b)·(shf−c) − d·sef'),
    dict(Model='SR-RESID-CUBE',Complexity=26,Form='srmed + a·shf·min(lf,b) − c·(max(d, cube(sef)) − e)'),
    dict(Model='SR-RESID-FULL',Complexity=30,Form='srmed + a·min(lf,b)·(shf−c) − d·(max(e, cube(sef)) − f) − g·lhf')])
summary['R²'] = summary['Model'].map({LABELS[n]:f'{r2[n]:.4f}' for n in ORDER})
summary.set_index('Model')
